# phase_2 — YOLO detector AUDIT (run after training)

Turns the YOLO concerns in `docs/critical_yolo.md` (= **B1**) into numbers, using the
trained `best.pt` + the built `yolo_ds` (NO M1/M3 needed). Tests:
1. per-class mAP · 2. **static-prior baseline** (YOLO must beat an image-blind mean-box
template) · 3. per-region IoU · 4. **stratified by anatomical atypicality** · 5. perturbation
(delete a region's pixels → does its box move?). The decisive **phep 3 (gold-vs-detector
oracle ablation at M3)** is DEFERRED — it needs M1 features + M3.

Run on a **GPU** session, *Run All*. Weights are pulled from Drive if not already local.

## CONFIG — edit these

In [10]:
import os
PHASE2_SRC = "/kaggle/working/repo/phase_2"
IMAGES      = "/kaggle/input/datasets/nguynnghin/mimic-cxr-448/mimic-cxr-448"
YOLO_LABELS = "/kaggle/input/datasets/nguynnghin/yolo-labels"
RUN_NAME    = "det29"
SPLIT       = "val"      # val (silver) | test (silver minus gold). Gold needs a separate ds.
MAX_IMAGES  = 2000       # cap IoU eval for speed (0 = all)
DRIVE_FOLDER_ID = "1c6AJ3fjsL449kiMK4xiXfnzwzA4gIo0O"
RCLONE_REMOTE   = "dhint:phase2_runs"
for k,v in dict(PHASE2_SRC=PHASE2_SRC,IMAGES=IMAGES,YOLO_LABELS=YOLO_LABELS,RUN_NAME=RUN_NAME,
                SPLIT=SPLIT,MAX_IMAGES=MAX_IMAGES,DRIVE_FOLDER_ID=DRIVE_FOLDER_ID,
                RCLONE_REMOTE=RCLONE_REMOTE).items():
    os.environ[k]=str(v)
print("config set | RUN_NAME =",RUN_NAME,"| SPLIT =",SPLIT)

config set | RUN_NAME = det29 | SPLIT = val


In [11]:
# code from GitHub (Internet ON)
!rm -rf /kaggle/working/repo && git clone -q https://github.com/hiennguyendang/phase_2_3_4_5 /kaggle/working/repo
!rm -rf /kaggle/working/phase_2 && cp -r "$PHASE2_SRC" /kaggle/working/phase_2
%cd /kaggle/working/phase_2
!pip install -q ultralytics

/kaggle/working/phase_2


In [12]:
import torch
print("torch:",torch.__version__,"| cuda:",torch.cuda.is_available())
assert torch.cuda.is_available(), "No GPU! Settings -> Accelerator -> GPU T4."

torch: 2.10.0+cu128 | cuda: True


## install rclone + auth (your OAuth token) + pull best.pt

In [13]:
%%bash
set -e
if ! command -v rclone >/dev/null 2>&1; then
  cd /kaggle/working
  curl -sLO https://downloads.rclone.org/rclone-current-linux-amd64.zip
  unzip -q -o rclone-current-linux-amd64.zip
  cp rclone-*-linux-amd64/rclone /usr/local/bin/ && chmod +x /usr/local/bin/rclone
fi
rclone version | head -1

rclone v1.74.3


In [14]:
import os
# Drive via YOUR OAuth token (NOT a service account). Secret GDRIVE_TOKEN from `rclone authorize "drive"`.
SYNC_OK="0"
try:
    from kaggle_secrets import UserSecretsClient
    sec=UserSecretsClient()
    os.environ["RCLONE_CONFIG_DHINT_TYPE"]="drive"
    os.environ["RCLONE_CONFIG_DHINT_TOKEN"]=sec.get_secret("GDRIVE_TOKEN").strip()
    os.environ["RCLONE_CONFIG_DHINT_SCOPE"]="drive"
    os.environ["RCLONE_CONFIG_DHINT_ROOT_FOLDER_ID"]=os.environ["DRIVE_FOLDER_ID"]
    os.environ.pop("RCLONE_CONFIG_DHINT_SERVICE_ACCOUNT_FILE",None)
    for ks,ke in [("GDRIVE_CLIENT_ID","RCLONE_CONFIG_DHINT_CLIENT_ID"),("GDRIVE_CLIENT_SECRET","RCLONE_CONFIG_DHINT_CLIENT_SECRET")]:
        try: os.environ[ke]=sec.get_secret(ks).strip()
        except Exception: pass
    if os.system('rclone lsd dhint: >/dev/null 2>&1')==0:
        SYNC_OK="1"; print("remote OK ->",os.environ["RCLONE_REMOTE"])
except Exception as e:
    print("[WARN] no Drive:",e)
os.environ["SYNC_OK"]=SYNC_OK; print("SYNC_OK =",SYNC_OK)

remote OK -> dhint:phase2_runs
SYNC_OK = 1


In [15]:
import os
# pull best.pt from Drive if not already local (same-session run already has it locally)
RUN=os.environ["RUN_NAME"]; W=f"/kaggle/working/runs/{RUN}/weights/best.pt"
if not os.path.exists(W) and os.environ.get("SYNC_OK")=="1":
    os.makedirs(os.path.dirname(W),exist_ok=True)
    os.system(f'rclone copy "{os.environ["RCLONE_REMOTE"]}/{RUN}/weights" /kaggle/working/runs/{RUN}/weights --include "best.pt"')
print("best.pt present:",os.path.exists(W),"->",W)
assert os.path.exists(W),"best.pt not found locally or on Drive — train first (or set RUN_NAME)."

best.pt present: True -> /kaggle/working/runs/det29/weights/best.pt


## assemble yolo_ds (symlink images to the prebuilt labels; gold held out)

In [16]:
import os

# Trỏ trực tiếp thư mục nhãn vào biến môi trường YOLO_LABELS
LDIR = os.environ["YOLO_LABELS"]
os.environ["LDIR"] = LDIR

# Chạy script tạo dataset
!python link_yolo_images.py --labels-dir "$LDIR" --images-root "$IMAGES" --out /kaggle/working/yolo_ds

exclude ids : 784 held-out stems (from /kaggle/working/phase_2/gold_ids.txt)
labels root : /kaggle/input/datasets/nguynnghin/yolo-labels/labels/labels
images root : /kaggle/input/datasets/nguynnghin/mimic-cxr-448/mimic-cxr-448
Indexing images (one walk, no PIL) ...
  images indexed: 377,105

[DONE] linked per split: {'train': 148242, 'val': 21335, 'test': 42194}
[HELD OUT] gold/excluded skipped per split: {'train': 0, 'val': 0, 'test': 784} (784 total — never linked into images/labels)
dataset.yaml -> /kaggle/working/yolo_ds/dataset.yaml


## run the audit

In [17]:
!python audit_yolo.py   --weights /kaggle/working/runs/$RUN_NAME/weights/best.pt   --yolo-ds /kaggle/working/yolo_ds --split $SPLIT --max-images $MAX_IMAGES   --out /kaggle/working/runs/$RUN_NAME/audit

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
weights : /kaggle/working/runs/det29/weights/best.pt
yolo-ds : /kaggle/working/yolo_ds
split   : val

[1] per-class mAP (model.val) ...
Ultralytics 8.4.80 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
Model summary (fused): 93 layers, 25,856,551 parameters, 0 gradients, 78.8 GFLOPs
val: Fast image access ✅ (ping: 0.4±0.2 ms, read: 6.1±1.1 MB/s, size: 44.7 KB)
val: Scanning /kaggle/working/yolo_ds/labels/val... 21335 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 21335/21335 186.1it/s 1:550.0sss
val: New cache created: /kaggle/working/yolo_ds/labels/val.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━

In [19]:
import os, json
RUN=os.environ["RUN_NAME"]
rep=f"/kaggle/working/runs/{RUN}/audit/audit_report.json"
print(open(rep).read())
if os.environ.get("SYNC_OK")=="1":
    os.system(f'rclone copy /kaggle/working/runs/{RUN}/audit "{os.environ["RCLONE_REMOTE"]}/{RUN}/audit"')
    print("pushed audit -> %s/%s/audit"%(os.environ["RCLONE_REMOTE"],RUN))

{
  "weights": "/kaggle/working/runs/det29/weights/best.pt",
  "split": "val",
  "n_images": 2000,
  "overall_map": {
    "mAP50": 0.9314150743099456,
    "mAP50_95": 0.6939654268772694,
    "precision": 0.9337516979646191,
    "recall": 0.8889534893213578
  },
  "overall_iou": {
    "iou_yolo": 0.8073,
    "iou_static": 0.4301,
    "gap": 0.3772
  },
  "per_class": {
    "abdomen": {
      "n": 1996,
      "iou_yolo": 0.8859,
      "iou_static": 0.689,
      "gap": 0.1969,
      "map": {
        "mAP50": 0.9896814490867177,
        "mAP50_95": 0.853900593300617
      }
    },
    "aortic arch": {
      "n": 1977,
      "iou_yolo": 0.7897,
      "iou_static": 0.2042,
      "gap": 0.5856,
      "map": {
        "mAP50": 0.9331178613087898,
        "mAP50_95": 0.6493955017201867
      }
    },
    "cardiac silhouette": {
      "n": 1999,
      "iou_yolo": 0.8141,
      "iou_static": 0.5389,
      "gap": 0.2752,
      "map": {
        "mAP50": 0.9605273468709892,
        "mAP50_95": 0.711